# Chapter 4 &mdash; The Pumping Lemma in Predicate Logic

**Concept 22 of the Chapter 4 decomposition:** *The Pumping Lemma in Predicate Logic, and a More General Version*

The quantified form &mdash; and a walk through the refutation in which you write every step and the solver is asked exactly one question.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Pumping-Lemma-Predicate-Logic/Concept-Pumping-Lemma-Predicate-Logic.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]



import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$$Reg(L) \Rightarrow \exists N : \forall w \in L : [\,|w|\ge N \Rightarrow \exists x,y,z :
w=xyz \wedge |xy|\le N \wedge y\ne\varepsilon \wedge \forall i\ge0 : xy^iz\in L\,]$$

Negated, every quantifier flips:

$$\neg Cond(L) \equiv \forall N : \exists w \in L : |w|\ge N \wedge \forall x,y,z :
[\,w=xyz \wedge |xy|\le N \wedge y\ne\varepsilon \Rightarrow \exists i : xy^iz\notin L\,]$$

Read off who chooses what: the **adversary** picks $N$ and the **split**; **you** pick
$w$ and $i$.

**You do the proof here.** This chapter refutes $Cond(L_{01})$ for
$L_{01}=\{0^i1^i\}$ on paper like this: take $w=0^N1^N$; note that $|xy|\le N$ forces
$y$ to lie inside the block of $0$s; pump, and watch the two counts part company.

Every one of those steps is **algebra over constants you introduce**. Write
$w = 0^M1^M$ and a split $x=0^X$, $y=0^Y$, $z=0^{M-X-Y}1^M$, and then

$$xy^2z \;=\; 0^{M+Y}1^M$$

is a *calculation*, not a search. So that is how this notebook checks your steps:
symbolically, instantly, with no solver anywhere near them. Get the concatenation
wrong, write an $x$ that is not $X$ symbols long, choose a $w$ you never tied to $N$
&mdash; each is caught as you type it.

Exactly one step is not algebra, and it is the last one: **can the pumped string
still be in $L$, for any admissible split at all?** That question &mdash; and only
that question &mdash; goes to z3, in a single call, at the very end.

So there are no worked splits to copy. Your $w$, your cases, your $i$, your
arithmetic. The solver just refuses to let you skip the ending.

## 2. Definitions

### The solver, and the algebra the walk runs on

In [ ]:
# --- z3 is not a Jove dependency; fetch it once --------------------------
try:
    import z3
except ImportError:
    import subprocess, sys
    print("installing z3 ...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'])
    import z3
import re
from itertools import product
print("z3", z3.get_version_string())

### Parametric strings

A *parametric string* is what you write on the board: `0^M 1^M`, or `0^(M-X-Y) 1^M`. Concatenating them, repeating one of them $i$ times, and comparing two of them are all **algebra** over the constants you introduced &mdash; no search, no solver.

In [ ]:
class Lin:
    # A linear integer expression: a constant plus integer multiples of
    # named constants.  Every exponent the student writes is one of these,
    # which is why the bookkeeping steps below need no solver at all.

    def __init__(self, const=0, terms=None):
        self.const = const
        self.terms = {k: v for k, v in (terms or {}).items() if v}

    @staticmethod
    def parse(t):
        out, t = Lin(), t.replace('-', '+-').replace(' ', '')
        for piece in t.split('+'):
            if not piece:
                continue
            sign = 1
            while piece.startswith('-'):
                sign, piece = -sign, piece[1:]
            m = re.fullmatch(r'(\d*)\*?([A-Za-z_]\w*)', piece)
            if m:
                out = out + Lin(0, {m.group(2): sign * int(m.group(1) or 1)})
            elif re.fullmatch(r'\d+', piece):
                out = out + Lin(sign * int(piece))
            else:
                raise ValueError('cannot read %r as a linear expression' % t)
        return out

    def __add__(self, o):
        d = dict(self.terms)
        for k, v in o.terms.items():
            d[k] = d.get(k, 0) + v
        return Lin(self.const + o.const, d)

    def __mul__(self, k):
        return Lin(self.const * k, {a: v * k for a, v in self.terms.items()})

    def __sub__(self, o):   return self + o * -1
    def __eq__(self, o):    return (isinstance(o, Lin) and self.const == o.const
                                    and self.terms == o.terms)
    def __hash__(self):     return hash((self.const, tuple(sorted(self.terms.items()))))
    def is_zero(self):      return self.const == 0 and not self.terms
    def names(self):        return set(self.terms)

    def z3(self, env):
        import z3
        e = z3.IntVal(self.const)
        for k, v in sorted(self.terms.items()):
            e = e + v * env[k]
        return z3.simplify(e)

    def __str__(self):
        p = []
        for k, v in sorted(self.terms.items(), key=lambda t: (t[1] < 0, t[0])):
            p.append(('-' if v < 0 else '+')
                     + ('' if abs(v) == 1 else '%d*' % abs(v)) + k)
        if self.const or not p:
            p.append(('-' if self.const < 0 else '+') + str(abs(self.const)))
        s = ''.join(p)
        return s[1:] if s.startswith('+') else s


class PStr:
    # A parametric string: runs of a single symbol, each carrying a Lin
    # exponent.  '0^M 1^M' and '0^(M-X-Y) 1^M' are both PStrs.  Adjacent
    # runs of the same symbol merge and provably-empty runs drop out, so
    # equality of two PStrs is exact string equality for every value of
    # the constants -- which is all the checking the walk needs until the
    # very last step.

    def __init__(self, runs):
        out = []
        for sym, e in runs:
            if e.is_zero():
                continue
            if out and out[-1][0] == sym:
                out[-1] = (sym, out[-1][1] + e)
            else:
                out.append((sym, e))
        self.runs = [(s, e) for s, e in out if not e.is_zero()]

    @staticmethod
    def parse(text):
        if text.strip() in ('', 'eps', 'epsilon', '""', "''"):
            return PStr([])
        runs = []
        for tok in text.split():
            m = (re.fullmatch(r'(\S)\^\((.+)\)', tok)
                 or re.fullmatch(r'(\S)\^(\S+)', tok))
            if m:
                runs.append((m.group(1), Lin.parse(m.group(2))))
            else:
                runs += [(ch, Lin(1)) for ch in tok]
        return PStr(runs)

    def __add__(self, o):  return PStr(self.runs + o.runs)
    def __eq__(self, o):   return isinstance(o, PStr) and self.runs == o.runs
    def __hash__(self):    return hash(tuple(self.runs))

    def repeat(self, i):
        out = []
        for _ in range(i):
            out += self.runs
        return PStr(out)

    def length(self):
        t = Lin()
        for _, e in self.runs:
            t = t + e
        return t

    def names(self):
        s = set()
        for _, e in self.runs:
            s |= e.names()
        return s

    def __str__(self):
        if not self.runs:
            return 'eps'
        out = []
        for s, e in self.runs:
            t = str(e)
            if t == '1':
                out.append(s)
            else:
                out.append('%s^%s' % (s, t if re.fullmatch(r'\w+', t)
                                      else '(%s)' % t))
        return ' '.join(out)

### A language

A shape, plus a condition on the block lengths. The shape says which strings are candidates; the condition does the counting that no regular expression could do.

In [ ]:
class Lang:
    # A language given as a SHAPE plus a condition on the block lengths.
    # Lang('0^a 1^b', lambda a, b: a == b) is 0^i 1^i: the shape says which
    # strings are even candidates, the condition does the counting that no
    # regular expression could do.

    def __init__(self, shape, cond, name=None):
        sh = PStr.parse(shape)
        self.blocks = []
        for s, e in sh.runs:
            if e.const or len(e.terms) != 1 or list(e.terms.values()) != [1]:
                raise ValueError('each block of the shape needs a plain name,'
                                 ' as in "0^a 1^b"; got %r' % str(e))
            self.blocks.append((s, list(e.terms)[0]))
        self.cond, self.name = cond, name or shape

    def _place(self, seq):
        # put the surviving runs onto the blocks, in order; None if impossible
        pos, out = 0, []
        for s in seq:
            if out and self.blocks[out[-1]][0] == s:
                out.append(out[-1])
                continue
            p = pos
            while p < len(self.blocks) and self.blocks[p][0] != s:
                p += 1
            if p == len(self.blocks):
                return None
            out.append(p)
            pos = p + 1
        return out

    def member(self, ps, env):
        # z3 constraint: this parametric string lies in this language.
        # A run whose exponent happens to be 0 is not there at all, so each
        # run may or may not survive; enumerate the possibilities and take
        # the Or.  With two or three runs that is a handful of disjuncts.
        import z3
        syms, opts = [s for s, _ in ps.runs], []
        for mask in product((0, 1), repeat=len(ps.runs)):
            keep = [i for i, m in enumerate(mask) if not m]
            where = self._place([syms[i] for i in keep])
            if where is None:
                continue
            tot = {b: Lin() for _, b in self.blocks}
            for pos, i in zip(where, keep):
                tot[self.blocks[pos][1]] = tot[self.blocks[pos][1]] + ps.runs[i][1]
            cs = [ps.runs[i][1].z3(env) == 0 for i, m in enumerate(mask) if m]
            cs += [ps.runs[i][1].z3(env) >= 1 for i in keep]
            cs.append(self.cond(*[tot[b].z3(env) for _, b in self.blocks]))
            opts.append(z3.And(*cs))
        return z3.Or(*opts) if opts else z3.BoolVal(False)

### The walk

In [ ]:
_OPS = ('<=', '>=', '!=', '==', '<', '>', '=')


def _split_rel(text):
    for op in _OPS:
        if op in text:
            l, r = text.split(op, 1)
            return Lin.parse(l), op, Lin.parse(r)
    raise ValueError('%r is not a relation like "M >= N"' % text)


def _rel(text, env):
    l, op, r = _split_rel(text)
    a, b = l.z3(env), r.z3(env)
    return {'<=': a <= b, '>=': a >= b, '!=': a != b,
            '==': a == b, '=': a == b, '<': a < b, '>': a > b}[op]


def _rel_names(text):
    l, _, r = _split_rel(text)
    return l.names() | r.names()


def _as_list(v):
    return [] if v is None else ([v] if isinstance(v, str) else list(v))


class PumpWalk:
    # Walk the refutation of Cond(L) the way Chapter 4 does it on paper.
    #
    # YOU supply every step, as a parametric string: the w, the split of w
    # into x y z for each case, and what xy^iz comes to.  Each of those is
    # algebra over the constants YOU introduced, and each is checked here on
    # the spot with no solver at all -- so there is no oracle to lean on and
    # no worked example to copy.
    #
    # The solver is asked exactly ONE question, by qed(), about the one step
    # algebra cannot settle: can any pumped string still be in L?

    TIMEOUT_MS = 10000

    def __init__(self, L, coach=True):
        self.L, self.w, self.ncases, self.coach = L, None, None, coach
        self.xlen, self.ylen = 'X', 'Y'
        self.decls, self.cases = [], {}
        print("Adversary: 'Suppose %s is regular.  Then it has a pumping"
              % L.name)
        print("            constant.  I have one in mind -- call it N.")
        print("            I am not telling you what it is.'")
        self._tip("",
                  "You: pick w in %s, as a PARAMETRIC string, long enough no"
                  % L.name,
                  "     matter which N the adversary meant.",
                  "     ->  choose_w('...')        for instance  '0^M 1^M'")

    def _tip(self, *lines):
        # the coaching.  Substantive output always prints; this does not,
        # once you have walked it through and want the replay short.
        if self.coach:
            for t in lines:
                print(t)

    # ---- you pick w ------------------------------------------------------
    def choose_w(self, text):
        w = PStr.parse(text)
        if not w.runs:
            print("w must be non-empty.")
            return self
        self.w, self.cases, self.ncases, self.decls = w, {}, None, []
        print("w = %s        |w| = %s" % (w, w.length()))
        new = sorted(w.names())
        if not new:
            print()
            print("That w carries no constant, so it is ONE fixed string --")
            print("and the adversary simply picks N longer than it.  Use a")
            print("constant, so that w can grow to meet any N.")
            return self
        print("Constants noted: %s." % ', '.join(new))
        self._tip("",
                  "The adversary fixed N BEFORE you chose w, so %s must be"
                  % ' and '.join(new),
                  "tied to N or your w may be too short.  State the tie:",
                  "   ->  declare('%s >= N')" % new[0])
        return self

    # ---- you tie your constants to N -------------------------------------
    def declare(self, *texts):
        if self.w is None:
            print("choose_w first.")
            return self
        for t in texts:
            _rel_names(t)                      # parse now, complain now
            self.decls.append(t)
            print("noted:  %s" % t)
        self._tip("",
                  "Into how many cases does the split of w into x y z fall?",
                  "   ->  splits(k)")
        return self

    # ---- you say how many cases ------------------------------------------
    def splits(self, k, x_len='X', y_len='Y'):
        if self.w is None:
            print("choose_w first.")
            return self
        self.ncases, self.cases = k, {}
        self.xlen, self.ylen = x_len, y_len
        self._tip(
            "Let %s = |x| and %s = |y|.  Those two ARE the split, so every"
            % (x_len, y_len),
            "case must be written in terms of them -- that is what makes a",
            "case stand for all the splits it covers, rather than for one.",
            "",
            "%d case%s to give:" % (k, '' if k == 1 else 's'),
            "   ->  case(1, x='...', y='...', z='...')")
        if k > 1:
            self._tip("",
                      "Overlapping cases are harmless.  A MISSING case is not;",
                      "qed() will hand you the split you left out.")
        return self

    # ---- you give one case ------------------------------------------------
    def case(self, k, x, y, z, when=None):
        if self.w is None:
            print("choose_w first.")
            return self
        X, Y, Z = (PStr.parse(t) for t in (x, y, z))
        bad = []
        if X.length() != Lin.parse(self.xlen):
            bad.append("|x| must be exactly %s; you wrote |x| = %s"
                       % (self.xlen, X.length()))
        if Y.length() != Lin.parse(self.ylen):
            bad.append("|y| must be exactly %s; you wrote |y| = %s"
                       % (self.ylen, Y.length()))
        if (X + Y + Z) != self.w:
            bad.append("x y z = %s, which is not w = %s" % (X + Y + Z, self.w))
        if bad:
            print("case %d REJECTED:" % k)
            for b in bad:
                print("   %s" % b)
            return self
        self.cases[k] = dict(x=X, y=Y, z=Z, when=_as_list(when),
                             i=None, pumped=None)
        print("case %d:  x = %s     y = %s     z = %s" % (k, X, Y, Z))
        print("   |x| = %s,  |y| = %s,  x y z = w.   Checked, by algebra."
              % (self.xlen, self.ylen))
        self._tip("   This case stands for every split whose pieces make",
                  "   sense, i.e. whose exponents stay >= 0.")
        for c in _as_list(when):
            print("   restricted further to:  %s" % c)
        if len(Y.runs) > 1:
            print("   note: this y straddles %d blocks of w." % len(Y.runs))
        self._tip("",
                  "Now pump it.  Pick i, and write out xy^iz yourself:",
                  "   ->  pump(%d, i=2, result='...')      pump up" % k,
                  "   ->  pump(%d, i=0, result='...')      pump down" % k)
        return self

    # ---- you pump it, and you write the result ----------------------------
    def pump(self, k, i, result):
        c = self.cases.get(k)
        if c is None:
            print("no case %d yet." % k)
            return self
        if i == 1:
            print("i = 1 gives back w, which is in L by construction.")
            print("Pump up (i >= 2) or down (i = 0).")
            return self
        want = c['x'] + c['y'].repeat(i) + c['z']
        got = PStr.parse(result)
        if got != want:
            print("case %d: that is not xy^%dz." % (k, i))
            print("   you wrote   %s" % got)
            print("   xy^%dz is    %s" % (i, want))
            print("   The algebra is not where the insight lives.  Fix it and")
            print("   carry on.")
            return self
        c['i'], c['pumped'] = i, want
        print("case %d, i = %d:   xy^%dz  =  %s" % (k, i, i, want))
        print("   matches x y^%d z.   Checked, by algebra -- no solver yet." % i)
        left = [j for j in range(1, (self.ncases or k) + 1)
                if self.cases.get(j) is None or self.cases[j]['pumped'] is None]
        if left:
            self._tip("", "Still owed: case(s) %s."
                      % ', '.join(str(j) for j in left))
        else:
            self._tip("",
                      "Every case now has a pumped string.  One question is",
                      "left, and it is the only one algebra cannot answer:",
                      "   ->  qed()")
        return self

    # ---- the one solver question -------------------------------------------
    def qed(self):
        import z3
        if self.w is None:
            print("choose_w first.")
            return self
        left = [k for k in range(1, (self.ncases or 0) + 1)
                if k not in self.cases or self.cases[k]['pumped'] is None]
        if left or not self.cases:
            print("case(s) %s still have no pumped string."
                  % (', '.join(str(k) for k in left) or 'all'))
            return self

        # -- one variable per constant of the walk; X and Y ARE the split,
        # -- so they are shared by every case, exactly as on paper.
        names = set(self.w.names()) | {'N', self.xlen, self.ylen}
        for t in self.decls:
            names |= _rel_names(t)
        for c in self.cases.values():
            names |= c['x'].names() | c['y'].names() | c['z'].names()
            for t in c['when']:
                names |= _rel_names(t)
        env = {n: z3.Int(n) for n in sorted(names)}
        X, Y, N = env[self.xlen], env[self.ylen], env['N']

        # X and Y range over the admissible splits throughout, so every
        # witness the solver hands back is a split that could really occur
        base = z3.And(N >= 1, X >= 0, Y >= 1, X + Y <= N,
                      *([e.z3(env) >= 0 for _, e in self.w.runs]
                        + [_rel(t, env) for t in self.decls]))

        def applies(c):
            # a case applies to exactly those splits where its own pieces
            # make sense -- every exponent it names is >= 0
            return z3.And(*([e.z3(env) >= 0
                             for _, e in c['x'].runs + c['y'].runs + c['z'].runs]
                            + [_rel(t, env) for t in c['when']] + [True]))

        # -- the obligations, each paired with what would BREAK it ----------
        checks = [('w is a member of %s' % self.L.name,
                   z3.Not(self.L.member(self.w, env))),
                  ('|w| >= N, whichever N the adversary meant',
                   self.w.length().z3(env) < N),
                  ('your %d case%s cover%s EVERY admissible split'
                   % (len(self.cases), '' if len(self.cases) == 1 else 's',
                      's' if len(self.cases) == 1 else ''),
                   z3.Not(z3.Or(*[applies(c)
                                  for c in self.cases.values()])))]
        for k in sorted(self.cases):
            c = self.cases[k]
            checks.append(('case %d: xy^%dz = %s falls OUTSIDE %s'
                           % (k, c['i'], c['pumped'], self.L.name),
                           z3.And(applies(c),
                                  self.L.member(c['pumped'], env))))

        # -- ONE query.  A flag per obligation, true exactly when it breaks --
        s = z3.Solver()
        s.set('timeout', self.TIMEOUT_MS)
        flags = []
        for n, (human, bad) in enumerate(checks):
            f = z3.Bool('broken_%d' % n)
            flags.append((f, human))
            s.add(f == z3.And(base, bad))
        s.add(z3.Or(*[f for f, _ in flags]))

        print("Obligations collected from your walk:")
        for _, human in flags:
            print("   *  %s" % human)
        print()
        print("ONE solver call: can any of them be broken?")
        r = s.check()

        if r == z3.unsat:
            print("   UNSAT -- not one of them can.")
            print()
            for _, human in flags:
                print("   holds:  %s" % human)
            print()
            print("That is !Cond(%s).  For every N there is a w in %s with"
                  % (self.L.name, self.L.name))
            print("|w| >= N such that EVERY admissible split of w has an i")
            print("putting xy^iz outside %s.  So no DFA has the pumping"
                  % self.L.name)
            print("property for %s, and %s is not regular.   QED"
                  % (self.L.name, self.L.name))
            return self
        if r == z3.unknown:
            print("   UNKNOWN -- the solver gave up.  That is not a proof, and")
            print("   it is not a refutation either.  Nothing is settled.")
            return self

        m = s.model()
        print("   SAT -- something breaks.  Here is one way:")
        print()
        for f, human in flags:
            if z3.is_true(m.eval(f, model_completion=True)):
                print("   BROKEN:  %s" % human)
        vals = ['%s = %s' % (n, m.eval(env[n], model_completion=True))
                for n in sorted(env)]
        print()
        print("   taking  %s" % ',  '.join(vals))
        print()
        print("   Your proof has a hole there.  Nothing is refuted yet.")
        return self

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;21.&nbsp;Regularity-Preserving Transformations Aid Proofs](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Regularity-Preserving-Transformations/Concept-Regularity-Preserving-Transformations.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4-DFA/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;1.&nbsp;Four Ways to Specify a Language, and Why You Need More Than One](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Four-Ways-To-Specify/Concept-Four-Ways-To-Specify.ipynb)&nbsp;&rarr;

---

## 3. Tests

**The algebra first.** This is the calculation Chapter 4 does on the board, done symbolically. Nothing here is a search.

In [ ]:
w = PStr.parse('0^M 1^M')
x, y, z = PStr.parse('0^X'), PStr.parse('0^Y'), PStr.parse('0^(M-X-Y) 1^M')

print('w        = %-16s  |w| = %s' % (w, w.length()))
print('x y z    = %-16s  is that w? %s' % (x + y + z, (x + y + z) == w))
print('x y^2 z  = %-16s  (pumped up)'   % (x + y.repeat(2) + z))
print('x y^0 z  = %-16s  (pumped down)' % (x + y.repeat(0) + z))
print()
print('0^(M+Y) 1^M came out of an addition. No solver was asked anything.')

**The language.** A shape the counting rides on. `0*1*` is a regular shape; `a == b` is the part no regular expression could express.

In [ ]:
L01 = Lang('0^a 1^b', lambda a, b: a == b, name='0^i 1^i')

def in_L(L, text):
    s = z3.Solver()
    s.add(L.member(PStr.parse(text), {}))
    return s.check() == z3.sat

for t in ['0^3 1^3', 'eps', '0^2 1^5', '0^4', '1^4', '1^2 0^2']:
    print('   %-10s in %s ?  %s' % (t, L01.name, in_L(L01, t)))

**The adversary opens.** Note that $N$ is never given a number. It stays a symbol, because the proof has to work for every $N$ at once.

In [ ]:
pl = PumpWalk(L01)

**Your $w$**, as a parametric string.

In [ ]:
pl.choose_w('0^M 1^M')

**Tie your constant to $N$.** The adversary moved first, so an $M$ floating free is an $M$ the adversary can out-run.

In [ ]:
pl.declare('M >= N')

**How many cases?** With $|xy|\le N\le M$, every admissible $y$ sits inside the $0$s &mdash; one case. That is what makes this $w$ a good choice.

In [ ]:
pl.splits(1)

**The split.** $X=|x|$ and $Y=|y|$ *are* the split, so writing the case in terms of them is what makes it stand for all the splits at once.

In [ ]:
pl.case(1, x='0^X', y='0^Y', z='0^(M-X-Y) 1^M')

**Pump it, and write the answer yourself.**

In [ ]:
pl.pump(1, i=2, result='0^(M+Y) 1^M')

**The one solver call.** Everything the walk recorded becomes an obligation; one query asks whether any of them can be broken.

In [ ]:
pl.qed()

**Pumping down closes it too.** Same walk, `i = 0`, replayed without the coaching.

In [ ]:
(PumpWalk(L01, coach=False)
   .choose_w('0^M 1^M')
   .declare('M >= N')
   .splits(1)
   .case(1, x='0^X', y='0^Y', z='0^(M-X-Y) 1^M')
   .pump(1, i=0, result='0^(M-Y) 1^M')
   .qed())

**Three ways to go wrong**, all caught by algebra before any solver runs &mdash; except the last, which is a real hole and needs the query to expose it.

In [ ]:
print('--- 1. the concatenation is wrong ------------------------------')
(PumpWalk(L01, coach=False).choose_w('0^M 1^M').declare('M >= N').splits(1)
   .case(1, x='0^X', y='0^Y', z='0^(M-X-Y) 1^M')
   .pump(1, i=2, result='0^(M+2*Y) 1^M'))

print()
print('--- 2. x is not written in terms of the split point ------------')
(PumpWalk(L01, coach=False).choose_w('0^M 1^M').declare('M >= N').splits(1)
   .case(1, x='0^(X+1)', y='0^Y', z='0^(M-X-1-Y) 1^M'))

print()
print('--- 3. w was never tied to N -----------------------------------')
(PumpWalk(L01, coach=False).choose_w('0^M 1^M').splits(1)
   .case(1, x='0^X', y='0^Y', z='0^(M-X-Y) 1^M')
   .pump(1, i=2, result='0^(M+Y) 1^M')
   .qed())

**Why $w$ has to be chosen well.** Tie $M$ to $N$ only loosely &mdash; $2M\ge N$, so $|w|\ge N$ still holds &mdash; and $y$ is no longer trapped in the $0$s. One case is now a *hole*, and the solver hands you the split you left out.

In [ ]:
print('=== |w| >= N, but only just: one case is not a proof ===========')
(PumpWalk(L01, coach=False)
   .choose_w('0^M 1^M').declare('2*M >= N', 'M >= 1').splits(1)
   .case(1, x='0^X', y='0^Y', z='0^(M-X-Y) 1^M')
   .pump(1, i=2, result='0^(M+Y) 1^M')
   .qed())

The same $w$ done honestly needs **all three** cases: $y$ inside the $0$s, $y$ straddling the boundary, $y$ inside the $1$s. It closes &mdash; but compare the work against the one-case walk above, and the point of picking $M\ge N$ is made for you.

In [ ]:
(PumpWalk(L01, coach=False)
   .choose_w('0^M 1^M').declare('2*M >= N', 'M >= 1').splits(3)
   .case(1, x='0^X',         y='0^Y',               z='0^(M-X-Y) 1^M')
   .case(2, x='0^X',         y='0^(M-X) 1^(X+Y-M)', z='1^(2*M-X-Y)')
   .case(3, x='0^M 1^(X-M)', y='1^Y',               z='1^(2*M-X-Y)')
   .pump(1, i=2, result='0^(M+Y) 1^M')
   .pump(2, i=2, result='0^M 1^(X+Y-M) 0^(M-X) 1^M')
   .pump(3, i=2, result='0^M 1^(M+Y)')
   .qed())

**The control that matters.** Run the identical walk on a language that *is* regular. If the machinery said QED here, it would be worthless.

In [ ]:
LREG = Lang('0^a 1^b', lambda a, b: z3.BoolVal(True), name='0* 1*')

(PumpWalk(LREG, coach=False)
   .choose_w('0^M 1^M').declare('M >= N').splits(1)
   .case(1, x='0^X', y='0^Y', z='0^(M-X-Y) 1^M')
   .pump(1, i=2, result='0^(M+Y) 1^M')
   .qed())

print()
print('Pumping a regular language just gives another member of it, so the')
print('last obligation cannot hold -- and no walk will ever make it hold.')

## 4. Exercises


1. Walk $\{0^n1^m : n<m\}$, written `Lang('0^a 1^b', lambda a, b: a < b)`. You will
   need a $w$ of your own. Then pump **up**, and afterwards try **down** instead.
   Only one direction closes. Why that one?
2. $\{0^n1^n0^n\}$ is `Lang('0^a 1^b 0^c', lambda a, b, c: z3.And(a == b, b == c))`.
   Take $w=0^M1^M0^M$ with $M\ge N$. How many cases does it need &mdash; and why is
   that *not* three?
3. In the three-case walk above, case 2 has a $y$ that straddles the boundary.
   Pump it **down** instead of up, work out $xy^0z$ by hand, and check that `qed()`
   still closes.
4. $\{0^n1^m : n\ne m\}$ is non-regular, but take $w=0^M1^{M+1}$ with $M\ge N$ and
   pump up by $2$: `qed()` comes back SAT, and hands you a surviving split. Read
   the witness. The lemma lets **each split choose its own $i$**, and this walk
   offers every split the same one. (The classic repair makes $|w|$ depend on $N!$.)
5. `case()` refuses an $x$ whose length is not exactly $X$. Say why in terms of what
   a *case* is supposed to stand for &mdash; and what a proof built out of cases that
   named their own lengths would actually have proved.
6. Change `declare('M >= N')` to `declare('M >= 1')` in the first walk. Which
   obligation breaks, and what witness comes back? Now find a *second* declaration
   that also breaks, and explain why the solver reported the one it did.
7. `qed()` distinguishes UNSAT from UNKNOWN. Construct a language where z3 returns
   UNKNOWN, and say exactly what you are entitled to conclude from it.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4-DFA/Concept-Pumping-Lemma-Predicate-Logic')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')